In [ ]:
import ipywidgets as widgets

supported_model_ids = [
    # SigLIP2
    "google/siglip2-base-patch16-224",
    "google/siglip2-base-patch16-256",
    "google/siglip2-base-patch16-naflex",
    "google/siglip2-base-patch32-256",
    "google/siglip2-base-patch16-384",
    "google/siglip2-base-patch16-512",
    "google/siglip2-large-patch16-256",
    "google/siglip2-large-patch16-384",
    "google/siglip2-large-patch16-512",
    "google/siglip2-so400m-patch14-224",
    "google/siglip2-so400m-patch14-384",
    "google/siglip2-so400m-patch16-256",
    "google/siglip2-so400m-patch16-384",
    "google/siglip2-so400m-patch16-512",
    "google/siglip2-so400m-patch16-naflex",
    "google/siglip2-giant-opt-patch16-256",
    "google/siglip2-giant-opt-patch16-384",
]


model_selector = widgets.Dropdown(options=supported_model_ids, value=supported_model_ids[0], description="Model:")
model_selector

In [ ]:
from transformers import AutoProcessor, AutoModel
import requests
from pathlib import Path


model_id = model_selector.value

print(f"Selected: {model_id}")

model = AutoModel.from_pretrained(model_id)
processor = AutoProcessor.from_pretrained(model_id)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image


def visualize_result(image: Image, labels: list[str], probs: np.ndarray, top: int = 5):
    """
    Utility function for visualization classification results
    params:
      image: input image
      labels: list of classification labels
      probs: model predicted softmaxed probabilities for each label
      top: number of the highest probability results for visualization
    returns:
      None
    """
    plt.figure(figsize=(72, 64))
    top_labels = np.argsort(-probs)[: min(top, probs.shape[0])]
    top_probs = probs[top_labels]
    plt.subplot(8, 8, 1)
    plt.imshow(image)
    plt.axis("off")

    plt.subplot(8, 8, 2)
    y = np.arange(top_probs.shape[-1])
    plt.grid()
    plt.barh(y, top_probs)
    plt.gca().invert_yaxis()
    plt.gca().set_axisbelow(True)
    plt.yticks(y, [labels[index] for index in top_labels])
    plt.xlabel("probability")

    print([{labels[x]: round(y, 2)} for x, y in zip(top_labels, top_probs)])

In [ ]:
import requests
from pathlib import Path
import torch
from PIL import Image

image_path = Path("test_image.jpg")
if not image_path.exists():
    r = requests.get(
        "http://images.cocodataset.org/val2017/000000039769.jpg",
    )

    with image_path.open("wb") as f:
        f.write(r.content)
image = Image.open(image_path)

input_labels = ["2 cats", "a plane", "a remote", "3 dogs"]
text_descriptions = [f"This is a photo of a {label}" for label in input_labels]

if "siglip2" in model_id:
    inputs = processor(text=text_descriptions, images=[image], padding=True, truncation=True, return_tensors="pt")
else:
    inputs = processor(text=text_descriptions, images=[image], padding="max_length", return_tensors="pt")

with torch.no_grad():
    results = model(**inputs)

logits_per_image = results["logits_per_image"]  # this is the image-text similarity score

probs = logits_per_image.softmax(dim=1).detach().numpy()
visualize_result(image, input_labels, probs[0])

In [ ]:
%pip install onnx
%pip install onnxruntime
%pip install optimum[onnx]

In [ ]:
!optimum-cli export onnx --help

In [ ]:
# Install Optimum ONNX exporter stack and export SigLIP2 to ONNX (feature-extraction)


!optimum-cli export onnx \
  --model google/siglip2-base-patch16-naflex \
  --task feature-extraction \
  ./siglip_text_onnx/

!find ./siglip_text_onnx -maxdepth 3 -type f | sort


In [ ]:
# Torch vs ONNX benchmark setup (SigLIP2 NaFlex vision encoder)

import shutil
import tempfile
import time
from pathlib import Path

import numpy as np
import torch

try:
    import onnxruntime as ort
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "onnxruntime is required. Install it in this notebook with: %pip install onnxruntime"
    ) from exc

BENCH_MODEL_ID = "google/siglip2-base-patch16-naflex"
LOCAL_ONNX_DIR = Path("./siglip_text_onnx")
WARMUP_RUNS = 10
BENCH_RUNS = 100
BATCH_SIZE = 8

if model_id != BENCH_MODEL_ID:
    print(f"Reloading Torch model for fair comparison: {BENCH_MODEL_ID}")
    model = AutoModel.from_pretrained(BENCH_MODEL_ID)
    processor = AutoProcessor.from_pretrained(BENCH_MODEL_ID)
    model_id = BENCH_MODEL_ID

model.eval()

if torch.cuda.is_available():
    device = torch.device("cuda")
    model = model.to(device)
else:
    device = torch.device("cpu")

print(f"Torch device: {device}")
print(f"Torch dtype: {getattr(model, 'dtype', 'unknown')}")
print(f"ORT providers available: {ort.get_available_providers()}")

if not LOCAL_ONNX_DIR.exists():
    raise FileNotFoundError(
        f"Local ONNX export directory not found: {LOCAL_ONNX_DIR}. Run the export cell first."
    )

# Find the ONNX model file from local export only.
model_candidates = [
    LOCAL_ONNX_DIR / "onnx" / "vision_model.onnx",
    LOCAL_ONNX_DIR / "vision_model.onnx",
    LOCAL_ONNX_DIR / "model.onnx",
]
source_onnx = next((p for p in model_candidates if p.exists()), None)
if source_onnx is None:
    raise FileNotFoundError(
        f"No ONNX model file found in {LOCAL_ONNX_DIR}. Tried: {[str(p) for p in model_candidates]}"
    )

# Copy to a physical runtime dir to avoid external-data path issues.
runtime_onnx_dir = Path(tempfile.mkdtemp(prefix="siglip2_onnx_runtime_"))
runtime_model_path = runtime_onnx_dir / source_onnx.name
shutil.copy2(source_onnx.resolve(), runtime_model_path)

# Copy external tensor data files if present.
for cand in [
    source_onnx.with_name(source_onnx.name + "_data"),
    source_onnx.with_name("vision_model.onnx_data"),
    source_onnx.with_name("model.onnx_data"),
]:
    if cand.exists():
        shutil.copy2(cand.resolve(), runtime_onnx_dir / cand.name)

providers = ["CPUExecutionProvider"]
if device.type == "cuda" and "CUDAExecutionProvider" in ort.get_available_providers():
    providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]

ort_sess_options = ort.SessionOptions()
ort_sess_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
ort_session = ort.InferenceSession(
    runtime_model_path.as_posix(),
    sess_options=ort_sess_options,
    providers=providers,
)

print(f"Local ONNX dir: {LOCAL_ONNX_DIR}")
print(f"Runtime ONNX dir: {runtime_onnx_dir}")
print(f"Runtime model path: {runtime_model_path}")
print(f"Using ORT providers: {ort_session.get_providers()}")
print(f"ONNX inputs: {[i.name for i in ort_session.get_inputs()]}")
print(f"ONNX outputs: {[o.name for o in ort_session.get_outputs()]}")


In [ ]:
# Build identical batched inputs for both runtimes

batch_images = [image.copy() for _ in range(BATCH_SIZE)]

# Match backend behavior for siglip2-naflex image encoding
inputs_cpu = processor(
    images=batch_images,
    return_tensors="pt",
    padding="max_length",
    max_num_patches=256,
)

torch_inputs = {}
for k, v in inputs_cpu.items():
    if v.dtype.is_floating_point and hasattr(model, "dtype"):
        torch_inputs[k] = v.to(device=device, dtype=model.dtype)
    else:
        torch_inputs[k] = v.to(device=device)

# Build ONNX inputs with exact dtypes expected by the ONNX graph.
ort_type_to_np = {
    "tensor(float)": np.float32,
    "tensor(float16)": np.float16,
    "tensor(double)": np.float64,
    "tensor(int64)": np.int64,
    "tensor(int32)": np.int32,
    "tensor(int16)": np.int16,
    "tensor(int8)": np.int8,
    "tensor(uint8)": np.uint8,
    "tensor(bool)": np.bool_,
}

onnx_inputs = {}
for inp in ort_session.get_inputs():
    name = inp.name
    if name not in inputs_cpu:
        continue

    arr = inputs_cpu[name].detach().cpu().numpy()
    target_dtype = ort_type_to_np.get(inp.type)
    if target_dtype is not None and arr.dtype != target_dtype:
        arr = arr.astype(target_dtype, copy=False)
    onnx_inputs[name] = arr

print({k: (tuple(v.shape), str(v.dtype)) for k, v in inputs_cpu.items()})
print({k: (tuple(v.shape), str(v.dtype)) for k, v in onnx_inputs.items()})
print({inp.name: inp.type for inp in ort_session.get_inputs()})


In [ ]:
# Benchmark helpers

def _sync_if_needed():
    if device.type == "cuda":
        torch.cuda.synchronize()


def _extract_embedding_tensor(out):
    if torch.is_tensor(out):
        return out

    for name in ("image_embeds", "pooler_output", "text_embeds"):
        if hasattr(out, name):
            t = getattr(out, name)
            if torch.is_tensor(t):
                return t

    if hasattr(out, "last_hidden_state") and torch.is_tensor(out.last_hidden_state):
        return out.last_hidden_state[:, 0, :]

    if isinstance(out, dict):
        for name in ("image_embeds", "pooler_output", "last_hidden_state", "text_embeds"):
            t = out.get(name)
            if torch.is_tensor(t):
                return t[:, 0, :] if name == "last_hidden_state" else t

    if isinstance(out, (tuple, list)):
        for t in out:
            if torch.is_tensor(t):
                return t

    raise TypeError(f"Unsupported torch output type for embedding extraction: {type(out)}")


def _vision_only_inputs(all_inputs):
    # SigLIP2 vision path keys.
    keys = ("pixel_values", "pixel_attention_mask", "spatial_shapes")
    return {k: all_inputs[k] for k in keys if k in all_inputs}


@torch.inference_mode()
def run_torch_once():
    image_inputs = _vision_only_inputs(torch_inputs)

    if hasattr(model, "get_image_features"):
        out = model.get_image_features(**image_inputs)
    elif hasattr(model, "vision_model"):
        out = model.vision_model(**image_inputs)
    else:
        # Last resort for non-SigLIP models.
        out = model(**image_inputs)

    emb = _extract_embedding_tensor(out)
    _sync_if_needed()
    return emb.detach().float().cpu().numpy()


def run_onnx_once():
    outputs = ort_session.run(None, onnx_inputs)
    return outputs[0]


def benchmark(fn, warmup=WARMUP_RUNS, runs=BENCH_RUNS, batch_size=BATCH_SIZE):
    for _ in range(warmup):
        fn()

    times_ms = []
    for _ in range(runs):
        t0 = time.perf_counter()
        fn()
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

    arr = np.asarray(times_ms, dtype=np.float64)
    mean_ms = float(arr.mean())
    return {
        "runs": runs,
        "warmup": warmup,
        "batch_size": batch_size,
        "mean_ms": mean_ms,
        "median_ms": float(np.percentile(arr, 50)),
        "p95_ms": float(np.percentile(arr, 95)),
        "p99_ms": float(np.percentile(arr, 99)),
        "std_ms": float(arr.std()),
        "min_ms": float(arr.min()),
        "max_ms": float(arr.max()),
        "samples_per_sec": float((1000.0 * batch_size) / mean_ms),
    }


def l2_normalize(x, eps=1e-12):
    return x / np.clip(np.linalg.norm(x, axis=-1, keepdims=True), eps, None)


In [ ]:
# Run benchmark + quick numeric parity check

torch_out = run_torch_once()
onnx_out = run_onnx_once()

print(f"Torch output shape: {torch_out.shape}")
print(f"ONNX output shape:  {onnx_out.shape}")

if torch_out.shape == onnx_out.shape:
    abs_diff = np.abs(torch_out - onnx_out)
    cos_sim = np.sum(l2_normalize(torch_out) * l2_normalize(onnx_out), axis=-1)
    print(f"Mean abs diff:      {abs_diff.mean():.6f}")
    print(f"Max abs diff:       {abs_diff.max():.6f}")
    print(f"Mean cosine sim:    {cos_sim.mean():.6f}")
    print(f"Min cosine sim:     {cos_sim.min():.6f}")
else:
    print("Skipping numeric parity because output shapes differ.")

torch_stats = benchmark(run_torch_once)
onnx_stats = benchmark(run_onnx_once)

print()
print("Torch stats")
for k, v in torch_stats.items():
    print(f"  {k}: {v}")

print()
print("ONNX stats")
for k, v in onnx_stats.items():
    print(f"  {k}: {v}")

if onnx_stats["mean_ms"] > 0:
    speedup = torch_stats["mean_ms"] / onnx_stats["mean_ms"]
    print()
    print(f"ONNX speedup vs Torch (mean latency): {speedup:.2f}x")
